In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, expr, rand, date_add, lit, floor, when
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)
from datetime import date


spark.sql("CREATE SCHEMA IF NOT EXISTS retailmart")
spark.sql("USE retailmart")
spark.sql("select current_catalog(), current_schema()").show()

In [0]:
"""
create sample RetailMaet Sales data
Simulates a real slow table with problems:
-too many small files
-no z-ordering
-only basic partitioning
"""

# Create or reuse the SparkSession for this notebook.
# Why this is used:
# - appName("RetailMart-SampleData-setup"): gives the Spark application a readable name in the UI/logs.
# - spark.sql.extensions=io.delta.sql.DefaultDeltaSparkSessionExtension:
#   enables Delta Lake SQL extensions so Delta features are available in the Spark session.
# - spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog:
#   tells Spark to use the Delta catalog implementation for the built-in spark_catalog,
#   which helps Spark understand Delta tables correctly.
# - getOrCreate(): returns the existing SparkSession if one already exists, otherwise creates a new one.
#
# Note: the correct config key is "spark.sql.extensions" (not "spark.sql.extenstions").
spark = SparkSession.builder.appName("RetailMart-SampleData-setup")\
    .config("spark.sql.extensions", "io.delta.sql.DefaultDeltaSparkSessionExtension")\
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
    .getOrCreate()

# ── Reference Data ───────────────────────────────────────────────── 
CITIES = [ "New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville" ]

CATEGORIES = [ "Electronics", "Clothing", "Furniture", "Sports", "Toys", "Books", "Grocery", "Beauty", "Automotive" ]

PAYMENT_METHODS = [ "Credit Card", "Debit Card", "PayPal", "Gift Card", "Cash" ] 

#---generate sample data-----
print("generating sample data")

# create base dataframe with 2million rows(simulate real values)
df_base= spark.range(0,2000000).toDF("sale_id")
# equivalent to df_base=df_base.withColumnRenamed("id","sale_id")

df_sales = (
    df_base
    .withColumn("sale_date", date_add(lit(date(2023, 1, 1)), (rand() * 365).cast("int")))
    .withColumn(
        "city",
        when(col("sale_id") % 12 == 0, lit("New York"))
        .when(col("sale_id") % 12 == 1, lit("Los Angeles"))
        .when(col("sale_id") % 12 == 2, lit("Chicago"))
        .when(col("sale_id") % 12 == 3, lit("Houston"))
        .when(col("sale_id") % 12 == 4, lit("Phoenix"))
        .when(col("sale_id") % 12 == 5, lit("Philadelphia"))
        .when(col("sale_id") % 12 == 6, lit("San Antonio"))
        .when(col("sale_id") % 12 == 7, lit("San Diego"))
        .when(col("sale_id") % 12 == 8, lit("Dallas"))
        .when(col("sale_id") % 12 == 9, lit("San Jose"))
        .when(col("sale_id") % 12 == 10, lit("Austin"))
        .otherwise(lit("Jacksonville"))
    )
    .withColumn(
        "product_category",
        when(col("sale_id") % 9 == 0, lit("Electronics"))
        .when(col("sale_id") % 9 == 1, lit("Clothing"))
        .when(col("sale_id") % 9 == 2, lit("Furniture"))
        .when(col("sale_id") % 9 == 3, lit("Sports"))
        .when(col("sale_id") % 9 == 4, lit("Toys"))
        .when(col("sale_id") % 9 == 5, lit("Books"))
        .when(col("sale_id") % 9 == 6, lit("Grocery"))
        .when(col("sale_id") % 9 == 7, lit("Beauty"))
        .otherwise(lit("Automotive"))
    )
    .withColumn("amount", (rand() * 990 + 10).cast("double"))  # $10 to $1000
    .withColumn("quantity", (rand() * 9 + 1).cast("int"))  # 1 to 10 units
    .withColumn("customer_id", (rand() * 50000 + 1).cast("int"))  # 50k unique customers
    .withColumn(
        "payment_method",
        when(col("sale_id") % 5 == 0, lit("Credit Card"))
        .when(col("sale_id") % 5 == 1, lit("Debit Card"))
        .when(col("sale_id") % 5 == 2, lit("PayPal"))
        .when(col("sale_id") % 5 == 3, lit("Gift Card"))
        .otherwise(lit("Cash"))
    )
    .withColumn("year", col("sale_date").cast("string").substr(1, 4))
)

# write table with problems(intentionally bad setup)
# this simulates what you find when investigate
# problems:
#     1. partitioned by only year(too broad)
#     2. no z-ordering
#     3. written in 50 small batches=many small files

# first write create table
df_sales.limit(40_000).write\
    .format("delta")\
    .mode("overwrite")\
    .partitionBy("year")\
    .saveAsTable("retailmart.sales_transactions")

# simulate 49 more small batch writes(create small file problem)
for batch in range(1,50):
    offset=batch*40_000
    df_sales.limit(offset+40_000)\
            .exceptAll(df_sales.limit(offset))\
            .write\
            .format("delta")\
            .mode("append")\
            .partitionBy("year")\
            .saveAsTable("retailmart.sales_transactions")  
    if(batch%10==0):
        print(f"written batch {batch}/49...")

print("Sample data generation completed")



In [0]:
# show what we created
spark.sql("""describe detail retailmart.sales_transactions""")\
    .select("format","numFiles","partitionColumns","sizeInBytes").show()


spark.sql("""
          select count(*) as total_rows from retailmart.sales_transactions
          """).show()

spark.sql("""
          select 
            year,
            count(*) as total_rows,
            count(distinct city) as cities,
            count(distinct product_category) as product_categories,
            count(distinct payment_method) as payment_methods
          from retailmart.sales_transactions
          group by year
          order by year asc
          """).show()